# Diverse Stochastic Image Inpainting with RePaint
**Model:** `google/ddpm-celebahq-256` · **Reference:** Lugmayr et al., CVPR 2022

This notebook preprocesses a custom image, creates a fixed binary mask, and samples multiple inpaintings with different random seeds. It expects your project to provide `src/` modules, including the RePaint pipeline.

> **Important:** RePaint's resampling jumps are part of the sampler implementation, not a generic option that can be added externally. The pipeline imported below must actually implement the RePaint forward-jump/resampling schedule and accept `jump_length` and `jump_n_sample`. If it does not, the notebook raises a helpful error rather than silently pretending to run full RePaint. The pretrained model is trained on CelebA-HQ faces, so results on unrelated image categories may be weak.

## Cell 1 — Environment, dependencies, GPU, and project path

Run this cell first. Kaggle/Colab usually already has PyTorch installed; the install cell avoids reinstalling it. `src/` is located dynamically by searching the current directory and `/kaggle/working` / `/content`.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Install only the requested supporting libraries. Keep the runtime's existing
# CUDA-enabled PyTorch build rather than accidentally replacing it with CPU Torch.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers", "torchmetrics", "matplotlib", "Pillow", "tqdm"
])

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. In Kaggle: Settings → Accelerator → GPU; "
        "in Colab: Runtime → Change runtime type → T4 GPU."
    )

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("PyTorch:", torch.__version__, flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("CUDA:", torch.version.cuda, flush=True)
print("Device:", DEVICE, flush=True)

def find_project_root(start: Path = Path.cwd()) -> Path:
    candidates = [start, *start.parents]
    for base in (Path("/kaggle/working"), Path("/content")):
        if base.exists():
            candidates.extend([base, *base.parents])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src").is_dir():
            return candidate
        # Common case: notebook is launched from a child directory.
        try:
            matches = list(candidate.glob("*/src"))
            if matches:
                return matches[0].parent.resolve()
        except (PermissionError, OSError):
            pass
    raise FileNotFoundError(
        "Could not find a project directory containing src/. "
        "Set PROJECT_ROOT manually to the folder that contains src/."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)
print("src exists:", (PROJECT_ROOT / "src").is_dir(), flush=True)

# Output directories
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DATA_DIR = OUTPUT_ROOT / "data" / "custom"
RESULTS_DIR = OUTPUT_ROOT / "repaint_results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Input directory:", DATA_DIR, flush=True)
print("Output directory:", RESULTS_DIR, flush=True)


## Cell 2 — Load, center-crop, and preprocess a custom image

Set `IMAGE_PATH` to an uploaded image. In Kaggle, use **Add Input** or upload the file into the notebook session; in Colab, upload it to the runtime and copy its path here. The image is converted to RGB, center-cropped to a square, resized to 256×256, and saved under `data/custom/`.

In [ ]:
# CHANGE THIS to your uploaded image path.
IMAGE_PATH = "/kaggle/working/my_image.jpg"
# Example alternatives:
# IMAGE_PATH = "/content/my_image.png"
# IMAGE_PATH = "/kaggle/input/my-dataset/example.jpg"

PREPROCESSED_PATH = DATA_DIR / "custom_256.png"

def preprocess_image(image_path: str | Path, output_path: str | Path,
                     size: int = 256) -> Image.Image:
    image_path = Path(image_path).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(
            f"Image not found: {image_path}\n"
            "Upload an image, then update IMAGE_PATH in this cell."
        )
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    width, height = image.size
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    image = image.crop((left, top, left + side, top + side))
    image = image.resize((size, size), Image.Resampling.LANCZOS)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path, format="PNG")
    print(f"Loaded {image_path} ({width}×{height}) → {output_path} (256×256 RGB)", flush=True)
    return image

ground_truth_pil = preprocess_image(IMAGE_PATH, PREPROCESSED_PATH)
ground_truth_np = np.asarray(ground_truth_pil)
plt.figure(figsize=(5, 5))
plt.imshow(ground_truth_np)
plt.title("Preprocessed ground truth (256×256 RGB)")
plt.axis("off")
plt.show()


## Cell 3 — Create and compare multiple mask types

Mask convention used throughout this notebook: **1 = known/visible pixels; 0 = missing pixels to inpaint**. The notebook supports several mask types:

- `center_box`: centered square hole (default 35% of image width/height)
- `wide_box`: wider rectangular hole
- `random_box`: randomly placed rectangular hole (reproducible via `MASK_SEED`)
- `multiple_boxes`: several smaller holes
- `brush`: irregular scratch/stroke-shaped missing regions
- `custom`: your own rectangle plus optional scratch strokes

Choose one mask with `SELECTED_MASK_TYPE`, or set `RUN_ALL_MASK_TYPES = True` to visualize and save every type. The stochastic RePaint loop in Cell 4 uses the selected mask. Running all mask types through sampling is controlled separately in Cell 4 to avoid unexpectedly multiplying GPU runtime.

If your `src/repaint_pipeline.py` uses the opposite mask convention, invert the mask before passing it to the pipeline.

In [ ]:
import random

# Choose: "center_box", "wide_box", "random_box", "multiple_boxes",
#         "brush", or "custom"
SELECTED_MASK_TYPE = "center_box"
RUN_ALL_MASK_TYPES = True  # visualize and save all mask types in this cell
MASK_SEED = 2026
BOX_FRACTION = 0.35

# Custom rectangle (left, top, right, bottom) in 256×256 pixel coordinates.
CUSTOM_BOX = (45, 55, 205, 190)
# Optional custom scratch strokes: (x1, y1, x2, y2, brush_width).
CUSTOM_SCRATCHES = [
    (35, 80, 215, 115, 18),
    (60, 200, 190, 150, 12),
]

MASK_TYPES = ["center_box", "wide_box", "random_box", "multiple_boxes", "brush", "custom"]

def make_mask(size=256, mode="center_box", box_fraction=0.35,
              custom_box=(45, 55, 205, 190), scratches=None, seed=2026):
    """Return float32 mask [H,W]: 1=known, 0=hole."""
    rng = random.Random(seed)
    mask_img = Image.new("L", (size, size), color=255)
    draw = ImageDraw.Draw(mask_img)

    if mode == "center_box":
        box_w = max(1, round(size * box_fraction))
        box_h = max(1, round(size * box_fraction))
        left, top = (size - box_w) // 2, (size - box_h) // 2
        draw.rectangle((left, top, left + box_w - 1, top + box_h - 1), fill=0)

    elif mode == "wide_box":
        box_w = round(size * 0.55)
        box_h = round(size * 0.25)
        left, top = (size - box_w) // 2, (size - box_h) // 2
        draw.rectangle((left, top, left + box_w - 1, top + box_h - 1), fill=0)

    elif mode == "random_box":
        box_w = max(1, round(size * box_fraction))
        box_h = max(1, round(size * box_fraction))
        left = rng.randint(0, size - box_w)
        top = rng.randint(0, size - box_h)
        draw.rectangle((left, top, left + box_w - 1, top + box_h - 1), fill=0)

    elif mode == "multiple_boxes":
        # Three separated holes with varied sizes and locations.
        boxes = [
            (int(size * 0.12), int(size * 0.16), int(size * 0.34), int(size * 0.34)),
            (int(size * 0.58), int(size * 0.18), int(size * 0.84), int(size * 0.38)),
            (int(size * 0.32), int(size * 0.62), int(size * 0.57), int(size * 0.83)),
        ]
        for box in boxes:
            draw.rectangle(box, fill=0)

    elif mode == "brush":
        # Organic free-form strokes. Randomized but repeatable with `seed`.
        for _ in range(7):
            x, y = rng.randint(15, size - 15), rng.randint(15, size - 15)
            points = [(x, y)]
            for _ in range(rng.randint(3, 6)):
                x = max(0, min(size - 1, x + rng.randint(-55, 55)))
                y = max(0, min(size - 1, y + rng.randint(-55, 55)))
                points.append((x, y))
            width = rng.randint(10, 24)
            draw.line(points, fill=0, width=width, joint="curve")
            radius = width // 2
            for px, py in (points[0], points[-1]):
                draw.ellipse((px-radius, py-radius, px+radius, py+radius), fill=0)

    elif mode == "custom":
        left, top, right, bottom = map(int, custom_box)
        left, right = sorted((max(0, min(size - 1, left)), max(0, min(size - 1, right))))
        top, bottom = sorted((max(0, min(size - 1, top)), max(0, min(size - 1, bottom))))
        draw.rectangle((left, top, right, bottom), fill=0)
        for stroke in (scratches or []):
            x1, y1, x2, y2, width = stroke
            draw.line((x1, y1, x2, y2), fill=0, width=int(width), joint="curve")
    else:
        raise ValueError(f"Unknown mask mode: {mode}. Choose from {MASK_TYPES}")

    return np.asarray(mask_img, dtype=np.float32) / 255.0

# Generate all mask variants once. The same selected mask is reused for every seed
# in Cell 4, making diversity comparisons fair (same image + same mask, different noise).
all_masks = {}
for mask_type in MASK_TYPES:
    all_masks[mask_type] = make_mask(
        size=256, mode=mask_type, box_fraction=BOX_FRACTION,
        custom_box=CUSTOM_BOX, scratches=CUSTOM_SCRATCHES, seed=MASK_SEED
    )

if SELECTED_MASK_TYPE not in all_masks:
    raise ValueError(f"SELECTED_MASK_TYPE must be one of {MASK_TYPES}")

mask_np = all_masks[SELECTED_MASK_TYPE]
masked_input_np = (ground_truth_np.astype(np.float32) * mask_np[..., None]).astype(np.uint8)

masks_to_show = MASK_TYPES if RUN_ALL_MASK_TYPES else [SELECTED_MASK_TYPE]
fig, axes = plt.subplots(len(masks_to_show), 3, figsize=(12, 3.8 * len(masks_to_show)))
if len(masks_to_show) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, mask_type in enumerate(masks_to_show):
    this_mask = all_masks[mask_type]
    masked = (ground_truth_np.astype(np.float32) * this_mask[..., None]).astype(np.uint8)
    axes[row, 0].imshow(ground_truth_np)
    axes[row, 0].set_title("Ground truth")
    axes[row, 1].imshow(this_mask, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title(f"{mask_type} mask (white=known)")
    axes[row, 2].imshow(masked)
    axes[row, 2].set_title(f"Masked input — {mask_type}")
    for ax in axes[row]:
        ax.axis("off")
    Image.fromarray((this_mask * 255).astype(np.uint8)).save(RESULTS_DIR / f"mask_{mask_type}.png")
    np.save(RESULTS_DIR / f"mask_{mask_type}.npy", this_mask)

plt.tight_layout()
plt.show()

# Cell 4 consumes these tensors. They are explicitly placed on CUDA.
mask_tensor = torch.from_numpy(mask_np.copy()).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
Image.fromarray(masked_input_np).save(RESULTS_DIR / f"masked_input_{SELECTED_MASK_TYPE}.png")
np.save(RESULTS_DIR / "mask.npy", mask_np)
print(f"Selected mask: {SELECTED_MASK_TYPE}", flush=True)
print(f"Known pixels: {mask_np.mean():.1%}; missing pixels: {(1-mask_np).mean():.1%}", flush=True)
print(f"Generated {len(all_masks)} mask types; saved mask images and arrays in {RESULTS_DIR}", flush=True)


## Cell 4 — Load the pretrained model and sample K RePaint variations

This cell uses your project's `src` implementation. Since project module names and function signatures vary, the import block tries the common locations shown below; if your repository uses different names, update only that block. The pipeline is expected to accept a normalized image tensor, a mask, `jump_length`, `jump_n_sample`, and a seeded `torch.Generator` (or equivalent seed argument).

**Tensor contract:** image tensor is `float32` in `[-1, 1]`, shape `[1, 3, 256, 256]`; mask tensor is `[1, 1, 256, 256]` with `1=known, 0=hole`. Output may be a tensor or PIL image. The adapter checks common output types and fails clearly for unsupported signatures. FP16 autocast is enabled on CUDA; keep model weights and all tensors on CUDA.

In [ ]:
import inspect
import importlib

MODEL_ID = "google/ddpm-celebahq-256"
K = 4
SEEDS = [1234 + i * 101 for i in range(K)]
JUMP_LENGTH = 10
JUMP_N_SAMPLE = 2

# ---- Project-specific imports: adapt these names to your src/ tree if needed. ----
repaint_pipeline = None
import_errors = []
for module_name in ("src.repaint_pipeline", "repaint_pipeline"):
    try:
        module = importlib.import_module(module_name)
        repaint_pipeline = getattr(module, "repaint_pipeline", None)
        if repaint_pipeline is None:
            # Some projects expose a class or a function named run_repaint.
            repaint_pipeline = getattr(module, "run_repaint", None)
        if repaint_pipeline is not None:
            print(f"Using pipeline: {module_name}.{repaint_pipeline.__name__}", flush=True)
            break
    except Exception as exc:
        import_errors.append(f"{module_name}: {type(exc).__name__}: {exc}")

if repaint_pipeline is None:
    raise ImportError(
        "Could not import repaint_pipeline from src.repaint_pipeline. "
        "Check that PROJECT_ROOT contains src/repaint_pipeline.py and that it "
        "exports repaint_pipeline (or run_repaint). Import attempts: "
        + " | ".join(import_errors)
    )

# Try common project loader names. A project may instead load the model inside its pipeline.
model = scheduler = None
loader_candidates = [
    ("src.model_loader", ("load_model_and_scheduler", "load_repaint_model", "load_model")),
    ("src.model", ("load_model_and_scheduler", "load_model")),
]
for module_name, function_names in loader_candidates:
    try:
        module = importlib.import_module(module_name)
        for function_name in function_names:
            loader = getattr(module, function_name, None)
            if loader is None:
                continue
            sig = inspect.signature(loader)
            kwargs = {}
            if "model_id" in sig.parameters: kwargs["model_id"] = MODEL_ID
            if "pretrained_model_name_or_path" in sig.parameters: kwargs["pretrained_model_name_or_path"] = MODEL_ID
            if "device" in sig.parameters: kwargs["device"] = DEVICE
            if "torch_dtype" in sig.parameters: kwargs["torch_dtype"] = torch.float16
            loaded = loader(**kwargs)
            if isinstance(loaded, dict):
                model = loaded.get("model", loaded.get("unet"))
                scheduler = loaded.get("scheduler")
            elif isinstance(loaded, (tuple, list)):
                if len(loaded) >= 2: model, scheduler = loaded[0], loaded[1]
                elif len(loaded) == 1: model = loaded[0]
            else:
                model = loaded
            print(f"Loaded model using {module_name}.{function_name}", flush=True)
            break
        if model is not None:
            break
    except Exception as exc:
        print(f"Loader candidate skipped ({module_name}): {exc}", flush=True)

# Fallback: load Hugging Face DDPM components directly. This is a model loader,
# not a replacement for the project's RePaint jump/resampling algorithm.
if model is None or scheduler is None:
    from diffusers import UNet2DModel, DDPMScheduler
    print(f"Loading pretrained checkpoint {MODEL_ID} from Hugging Face...", flush=True)
    model = UNet2DModel.from_pretrained(MODEL_ID).to(device=DEVICE, dtype=torch.float16)
    scheduler = DDPMScheduler.from_pretrained(MODEL_ID)
    print("Pretrained UNet and DDPM scheduler loaded.", flush=True)

# Strict device placement and eval mode.
if hasattr(model, "to"):
    model = model.to(DEVICE)
if hasattr(model, "eval"):
    model.eval()
if hasattr(scheduler, "set_timesteps"):
    # RePaint implementations often configure their own timesteps. Do not set
    # them here unless your pipeline expects the scheduler preconfigured.
    pass

image_tensor = torch.from_numpy(ground_truth_np.copy()).permute(2, 0, 1).unsqueeze(0)
image_tensor = (image_tensor.float() / 127.5 - 1.0).to(DEVICE)
mask_tensor = torch.from_numpy(mask_np.copy()).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
assert image_tensor.device.type == "cuda" and mask_tensor.device.type == "cuda"
assert image_tensor.shape == (1, 3, 256, 256)
assert mask_tensor.shape == (1, 1, 256, 256)

def _to_device(obj):
    if torch.is_tensor(obj):
        return obj.to(DEVICE)
    return obj

def _invoke_repaint(pipeline_fn, model, scheduler, image, mask, generator, seed):
    """Call a project's pipeline using only parameter names it declares."""
    sig = inspect.signature(pipeline_fn)
    params = sig.parameters
    available = {
        "model": model, "unet": model, "scheduler": scheduler,
        "image": image, "image_tensor": image, "image_0": image, "x0": image,
        "original_image": image, "ground_truth": image,
        "mask": mask, "mask_tensor": mask,
        "generator": generator, "torch_generator": generator,
        "seed": seed, "random_seed": seed,
        "device": DEVICE, "jump_length": JUMP_LENGTH,
        "jump_n_sample": JUMP_N_SAMPLE, "num_inference_steps": 1000,
    }
    kwargs = {name: available[name] for name in params if name in available}
    required_unknown = [
        name for name, p in params.items()
        if p.default is inspect.Parameter.empty
        and p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)
        and name not in kwargs
    ]
    if required_unknown:
        raise TypeError(
            f"Pipeline has required parameters not covered by this adapter: {required_unknown}. "
            f"Signature is {sig}. Update _invoke_repaint() to match your src API."
        )
    # Explicitly ensure the pipeline supports the RePaint resampling controls.
    if "jump_length" not in params or "jump_n_sample" not in params:
        raise TypeError(
            f"Pipeline signature {sig} does not expose jump_length and jump_n_sample. "
            "This call cannot guarantee RePaint forward resampling jumps. Update your "
            "src/repaint_pipeline.py to implement those controls and expose them."
        )
    return pipeline_fn(**kwargs)

def output_to_rgb_array(output):
    if isinstance(output, dict):
        for key in ("image", "sample", "result", "images", "x0"):
            if key in output:
                output = output[key]
                break
    if isinstance(output, (tuple, list)):
        output = output[0]
    if isinstance(output, Image.Image):
        return np.asarray(output.convert("RGB").resize((256, 256))).astype(np.uint8)
    if not torch.is_tensor(output):
        raise TypeError(f"Unsupported pipeline output type: {type(output)}")
    output = output.detach().float().cpu()
    if output.ndim == 4: output = output[0]
    if output.ndim == 3 and output.shape[0] in (1, 3, 4):
        output = output[:3].permute(1, 2, 0)
    arr = output.numpy()
    # Handle either [-1,1] or [0,1] model output ranges.
    if arr.min() < -0.05:
        arr = (arr + 1.0) / 2.0
    arr = np.clip(arr, 0.0, 1.0)
    return (arr * 255.0).round().astype(np.uint8)

variations = []
for index, seed in enumerate(SEEDS, start=1):
    print(f"\n[{index}/{K}] Sampling with seed={seed}", flush=True)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    torch.cuda.empty_cache()
    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = _invoke_repaint(
                repaint_pipeline, model, scheduler,
                image_tensor, mask_tensor, generator, seed
            )
    result = output_to_rgb_array(output)
    variations.append(result)
    save_path = RESULTS_DIR / f"repaint_seed_{seed}.png"
    Image.fromarray(result).save(save_path)
    print(f"Saved: {save_path}", flush=True)

print(f"Finished {len(variations)} samples.", flush=True)


## Cell 5 — Comparative grid, high-resolution exports, and ZIP archive

The ground truth and masked input are included for reference. Each variation is labeled with its seed so results are reproducible. The saved PNGs are the individual outputs; the figure is exported as both PNG and PDF.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

if len(variations) != K:
    raise RuntimeError(f"Expected {K} variations, got {len(variations)}.")

panels = [("Ground truth", ground_truth_np), ("Masked input", masked_input_np)]
panels += [(f"RePaint seed {seed}", image) for seed, image in zip(SEEDS, variations)]

ncols = min(3, len(panels))
nrows = int(np.ceil(len(panels) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.5 * nrows))
axes = np.asarray(axes).reshape(-1)
for ax, (title, image) in zip(axes, panels):
    ax.imshow(image)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
for ax in axes[len(panels):]:
    ax.axis("off")
fig.suptitle("RePaint stochastic inpainting variations", fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.97))

grid_png = RESULTS_DIR / "comparison_grid.png"
grid_pdf = RESULTS_DIR / "comparison_grid.pdf"
fig.savefig(grid_png, dpi=300, bbox_inches="tight")
fig.savefig(grid_pdf, bbox_inches="tight")
plt.show()
print("Saved grid:", grid_png, flush=True)
print("Saved PDF:", grid_pdf, flush=True)

# Save metadata for reproducibility.
metadata_path = RESULTS_DIR / "run_metadata.txt"
with open(metadata_path, "w", encoding="utf-8") as f:
    f.write(f"model_id={MODEL_ID}\n")
    f.write(f"image_path={PREPROCESSED_PATH}\n")
    f.write(f"mask_mode={MASK_MODE}\n")
    f.write(f"box_fraction={BOX_FRACTION}\n")
    f.write(f"seeds={SEEDS}\n")
    f.write(f"jump_length={JUMP_LENGTH}\n")
    f.write(f"jump_n_sample={JUMP_N_SAMPLE}\n")
    f.write("mask_convention=1 known, 0 missing\n")

zip_path = OUTPUT_ROOT / "repaint_inpainting_outputs.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=f"repaint_results/{path.name}")

print("\nAll done.", flush=True)
print("Individual images and figures:", RESULTS_DIR, flush=True)
print("ZIP archive:", zip_path, flush=True)
print("Files:", flush=True)
for path in sorted(RESULTS_DIR.iterdir()):
    print(f"  {path.name} ({path.stat().st_size / 1024:.1f} KB)", flush=True)
